In [ ]:
print("--- Exportando resultados ---\n")

# Crear directorio de salida
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

# 1. Exportar análisis de red
extractor.export_network_analysis(output_dir=str(output_dir), G=G, analysis=analysis)

# 2. Guardar insights en JSON
insights_file = output_dir / "network_insights.json"
with open(insights_file, 'w', encoding='utf-8') as f:
    json.dump(insights, f, indent=2)
print(f"✓ network_insights.json")

# 3. Exportar también el análisis completo de tweets (de U4)
extractor.export_results(output_dir=str(output_dir))

# 4. Crear resumen ejecutivo
summary = f"""
AEC3 - ANÁLISIS DE REDES SOCIALES Y LLM
========================================

INSIGHTS CLAVE:
- Top 3 nodos influyentes: {', '.join(top_3_influential)}
- Hashtags principales: {', '.join(top_hashtags)}
- Comunidades detectadas: {communities_info.get('num_communities', 0)}

ESTADÍSTICAS DE RED:
- Total de nodos: {G.number_of_nodes()}
- Total de aristas: {G.number_of_edges()}
- Densidad: {nx.density(G):.4f}
- Tweets procesados: {len(extractor.data):,}
- Usuarios únicos: {extractor.data['user_name'].nunique():,}

ANÁLISIS DEL LLM:
{llm_response}

Archivos generados:
- network_stats.json: Estadísticas detalladas del grafo
- communities.json: Comunidades identificadas
- network_prompt.txt: Prompt enviado al LLM
- network_graph.gexf: Grafo en formato Gephi
- network_insights.json: Insights clave
- llm_response.txt: Respuesta completa del LLM
"""

summary_file = output_dir / "RESUMEN_EJECUTIVO.txt"
summary_file.write_text(summary, encoding='utf-8')
print(f"✓ RESUMEN_EJECUTIVO.txt")

print(f"\n✅ Todos los resultados guardados en: {output_dir.resolve()}")

# Listar archivos generados
print(f"\nArchivos generados:")
for file in sorted(output_dir.glob("**/*")):
    if file.is_file():
        size = file.stat().st_size
        print(f"  - {file.relative_to(output_dir)} ({size:,} bytes)")

## 9. Export Network Analysis Results

Guardamos todos los resultados: métricas, grafo, comunidades, prompts y respuestas del LLM.

In [ ]:
print("--- Generando respuesta del LLM ---\n")

if model is not None and tokenizer is not None:
    print("ℹ Usando modelo local...\n")
    
    # Preparar el prompt
    generate_prompt = prompt[:500]  # Limitar tamaño del prompt de entrada
    
    # Tokenizar
    inputs = tokenizer(generate_prompt, return_tensors="pt").to(device)
    input_length = inputs['input_ids'].shape[1]
    
    print(f"Longitud del prompt: {input_length} tokens")
    print(f"Generando respuesta (esto puede tardar 30-60 segundos)...\n")
    
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.7,
                top_p=0.95,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Decodificar respuesta
        response_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        llm_response = response_text[len(generate_prompt):].strip()
        
        print("✓ Respuesta generada:")
        print("=" * 80)
        print(llm_response)
        print("=" * 80)
        
    except Exception as e:
        print(f"✗ Error generando respuesta: {e}")
        llm_response = "[Error generando respuesta]"
else:
    print("ℹ Usando API alternativa (Google)...\n")
    
    try:
        import google.generativeai as genai
        
        api_key = os.environ.get("GOOGLE_API_KEY", None)
        if api_key:
            genai.configure(api_key=api_key)
            model_gen = genai.GenerativeModel('gemini-pro')
            
            print("Consultando Google Gemini...\n")
            response = model_gen.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(
                    max_output_tokens=256,
                    temperature=0.7
                )
            )
            
            llm_response = response.text
            print("✓ Respuesta de Google Gemini:")
            print("=" * 80)
            print(llm_response)
            print("=" * 80)
        else:
            print("⚠ No se pudo usar Google AI (GOOGLE_API_KEY no configurada)")
            llm_response = "[Sin modelo disponible]"
    except ImportError:
        print("⚠ google-generativeai no instalado")
        llm_response = "[Sin modelo disponible]"

# Guardar respuesta
response_file = Path("llm_response.txt")
response_file.write_text(llm_response, encoding='utf-8')
print(f"\n✓ Respuesta guardada en: {response_file}")

## 8. Chat Interactivo with LLM

Ejecutar el modelo para obtener análisis interpretativo del grafo basado en el prompt generado.

In [ ]:
print("--- Inicializando modelo LLM local ---\n")

# Opciones de modelos (en orden de preferencia/facilidad de uso)
model_options = [
    "google/gemma-2b-it",          # Más pequeño, cabe en GPU modesta
    "facebook/opt-6.7b",            # Meta OPT modelo
    "mistralai/Mistral-7B-v0.1",   # Mistral (requiere más VRAM)
]

# Seleccionar modelo disponible
selected_model = model_options[0]  # Usar Gemma 2B por defecto
print(f"Modelo seleccionado: {selected_model}")

# Configuración
use_gpu = torch.cuda.is_available()
device = "cuda" if use_gpu else "cpu"

print(f"Device: {device}")
if use_gpu:
    print(f"GPU disponible: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("\nℹ Nota: La descarga del modelo puede tardar algunos minutos (1-5 GB)")
print("  Esto sucede solo la primera vez, luego se cachea localmente.\n")

try:
    print(f"Descargando tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(selected_model)
    print(f"✓ Tokenizer cargado")
    
    print(f"Descargando modelo (puede tomar tiempo)...")
    model = AutoModelForCausalLM.from_pretrained(
        selected_model,
        torch_dtype=torch.float16 if use_gpu else torch.float32,
        device_map="auto" if use_gpu else None,
    )
    
    if device == "cpu":
        model = model.to(device)
    
    print(f"✓ Modelo cargado correctamente")
    print(f"Parámetros del modelo: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
    
except Exception as e:
    print(f"✗ Error cargando modelo local: {e}")
    print(f"  Usando alternativa: Google Generative AI API")
    selected_model = None
    model = None
    tokenizer = None

## 7. Initialize and Test Local LLM

Cargamos el modelo preentrenado de Hugging Face (Gemma o alternativa) y configuramos para generación de texto.

In [ ]:
print("--- Generando prompt contextualizado para LLM ---\n")

# Generar prompt automáticamente
prompt = extractor.generate_prompt_from_network(G, analysis)

print("PROMPT GENERADO:")
print("=" * 80)
print(prompt)
print("=" * 80)

# Guardar el prompt para referencia
prompt_file = Path("generated_prompt.txt")
prompt_file.write_text(prompt, encoding='utf-8')
print(f"\n✓ Prompt guardado en: {prompt_file}")

## 6. Generate Prompt from Network Analysis

Transformamos las métricas técnicas del grafo en un prompt narrativo que el LLM puede interpretar.

In [ ]:
print("--- Extrayendo insights clave ---\n")

# Top 3 nodos por centralidad
top_3_influential = [node for node, _ in analysis['top_degree_centrality'][:3]]
print(f"Top 3 nodos influentes: {top_3_influential}")

# Hashtags más frecuentes
hashtags_stats = extractor.analytics_hashtags_extended()
top_hashtags = hashtags_stats['overall'].head(5)['hashtag'].tolist()
print(f"Top 5 hashtags: {top_hashtags}")

# Estadísticas de tweets
total_tweets = len(extractor.data)
unique_users = extractor.data['user_name'].nunique()
print(f"\nTweets totales: {total_tweets}")
print(f"Usuarios únicos: {unique_users}")
print(f"Promedio tweets por usuario: {total_tweets / unique_users:.2f}")

# Análisis de sentimiento (si estaba hecho en U4)
if 'sentiment_polarity' in extractor.data.columns:
    avg_polarity = extractor.data['sentiment_polarity'].mean()
    avg_subjectivity = extractor.data['sentiment_subjectivity'].mean()
    print(f"\nSentimiento promedio:")
    print(f"  - Polaridad: {avg_polarity:.4f} (negativo: -1, positivo: +1)")
    print(f"  - Subjetividad: {avg_subjectivity:.4f} (objetivo: 0, subjetivo: 1)")
else:
    print("\nℹ (Análisis de sentimiento no disponible)")

# Crear diccionario de insights
insights = {
    'top_3_influential_nodes': top_3_influential,
    'top_hashtags': top_hashtags,
    'global_stats': {
        'total_tweets': int(total_tweets),
        'unique_users': int(unique_users),
        'network_nodes': G.number_of_nodes(),
        'network_edges': G.number_of_edges(),
        'network_density': float(nx.density(G))
    },
    'communities': {
        'num_communities': communities_info.get('num_communities', 0),
        'sizes': communities_info.get('sizes', [])
    }
}

print("\n✓ Insights extraídos y almacenados")

## 5. Extract Key Network Insights

Extrae los insights más relevantes que serán utilizados para generar el prompt del LLM.

In [ ]:
print("--- Analizando métricas de red ---\n")

# Ejecutar análisis completo
analysis = extractor.analyze_network(G, top_k=5)

# Mostrar top nodos por diferentes centralidades
print(f"TOP 5 NODOS POR CENTRALIDAD DE GRADO:")
for i, (node, score) in enumerate(analysis['top_degree_centrality'], 1):
    print(f"  {i}. {node}: {score:.4f}")

print(f"\nTOP 5 NODOS POR CENTRALIDAD DE INTERMEDIACIÓN (betweenness):")
for i, (node, score) in enumerate(analysis['top_betweenness_centrality'], 1):
    print(f"  {i}. {node}: {score:.4f}")

print(f"\nTOP 5 NODOS POR CENTRALIDAD DE CERCANÍA (closeness):")
for i, (node, score) in enumerate(analysis['top_closeness_centrality'], 1):
    print(f"  {i}. {node}: {score:.4f}")

# Comunidades
communities_info = analysis['communities']
print(f"\n--- DETECCIÓN DE COMUNIDADES ---")
print(f"Número de comunidades: {communities_info.get('num_communities', 0)}")
print(f"Tamaños de comunidades: {communities_info.get('sizes', [])}")

# Visualizar distribución de grados
degrees = [G.degree(n) for n in G.nodes()]
plt.figure(figsize=(10, 5))
plt.hist(degrees, bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Grado del nodo')
plt.ylabel('Frecuencia')
plt.title('Distribución de grados en la red')
plt.grid(True, alpha=0.3)
plt.show()

print("✓ Análisis completado")

## 4. Analyze Network Metrics and Detect Communities

Calculamos métricas de centralidad (qué nodos son más importantes/influyentes) y detectamos comunidades (grupos densamente conectados).

In [ ]:
print("--- Construyendo grafo de interacciones ---\n")

# Construir el grafo
G = extractor.build_interaction_graph(mention_extraction=True)

# Información básica del grafo
print(f"Estadísticas del Grafo:")
print(f"  - Nodos totales: {G.number_of_nodes()}")
print(f"  - Aristas totales: {G.number_of_edges()}")
print(f"  - Densidad: {nx.density(G):.4f}")
print(f"  - ¿Conectado?: {nx.is_connected(G.to_undirected())}")

# Desglose de tipos de nodos
node_types = {}
for node, attrs in G.nodes(data=True):
    ntype = attrs.get('node_type', 'unknown')
    node_types[ntype] = node_types.get(ntype, 0) + 1

print(f"\nTipos de nodos:")
for ntype, count in node_types.items():
    print(f"  - {ntype}: {count}")

# Mostrar algunos nodos aleatorios
print(f"\nEjemplos de nodos:")
for node in list(G.nodes())[:5]:
    print(f"  - {node}")

## 3. Build Interaction Graph from Mentions

Construimos un grafo dirigido donde:
- **Nodos** = Usuarios y hashtags
- **Aristas** = Menciones (@usuario) o uso compartido de hashtags
- **Peso de arista** = Frecuencia de la interacción

In [ ]:
# Configurar ruta del dataset
# Buscar el dataset de Bitcoin from Unidad 4
data_paths = [
    "../Unit_4/AEC2/Bitcoin_tweets.csv",
    "Bitcoin_tweets.csv",
    "../../../5th_Course/Data_Engineering_Project/data/tweets.csv"
]

data_file = None
for path in data_paths:
    if Path(path).exists():
        data_file = path
        print(f"✓ Dataset encontrado: {path}")
        break

if data_file is None:
    print("⚠ Dataset no encontrado. Usando archivo de ejemplo.")
    # Si no hay dataset, creamos un pequeño conjunto de prueba
    data_file = "sample_tweets.csv"
    print(f"Nota: Debes colocar un archivo CSV con columnas 'user_name', 'date', 'text' en la carpeta")
else:
    print(f"Usando dataset: {data_file}")

# Instanciar DataExtractor
extractor = DataExtractor(source=data_file)

# Cargar datos
print("\n--- Cargando datos ---")
extractor.load_data()

# Mostrar estadísticas básicas
print(f"\n--- Estadísticas del Dataset ---")
print(f"Tweets totales: {len(extractor.data):,}")
print(f"Usuarios únicos: {extractor.data['user_name'].nunique():,}")
print(f"Rango de fechas: {extractor.data['date'].min()} - {extractor.data['date'].max()}")
print(f"\nPrimeros 3 tweets:")
print(extractor.data[['user_name', 'text']].head(3).to_string())

## 2. Initialize DataExtractor and Load Data

In [ ]:
# Importar la clase DataExtractor
from data_extractor import DataExtractor

print("✓ DataExtractor importada correctamente")

In [ ]:
import sys
import os
from pathlib import Path

# Agregar el directorio actual al path para importar data_extractor
sys.path.insert(0, str(Path.cwd()))

# Verificar disponibilidad de CUDA
import torch
print(f"✓ PyTorch versión: {torch.__version__}")
print(f"✓ CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  - GPU: {torch.cuda.get_device_name(0)}")
    print(f"  - CUDA versión: {torch.version.cuda}")
else:
    print("  ⚠ GPU no disponible, se usará CPU (más lento)")

# Librerías de datos y cálculo
import pandas as pd
import numpy as np
import json

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Network analysis
import networkx as nx
from networkx.algorithms import community

# NLP y análisis
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from textblob import TextBlob
import gensim
from gensim.corpora import Dictionary
from gensim.models import LdaModel

# LLM y Transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
import logging

# Descargar recursos de NLTK
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Configurar estilo de gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("\n✅ Todas las librerías importadas correctamente")

## 1. Import Required Libraries

# AEC3 - Análisis de Redes Sociales + Integración LLM
## Unidad 6: Extensión de la práctica AEC2 con NetworkX y Transformers

**Objetivo**: Combinar análisis de redes sociales con interpretación de LLM local para obtener insights cualitativos sobre interacciones en Twitter.

**Componentes principales**:
1. Análisis de redes con NetworkX (grafo de interacciones, centralidad, comunidades)
2. Generación de prompts a partir de métricas de red
3. Integración de LLM local (Gemma) para análisis interpretativo
4. Exportación de resultados